In [38]:
import networkx as nx
from rdflib import Graph, BNode, URIRef, Literal, RDF, Namespace
import rdflib
from pyvis import network as net
from tabulate import tabulate

In [51]:
# define my own schema.org namespace, since it is a bit broken in the data atm (I have fixed this, but it will only work, once the data are updated
SCHEMA = Namespace("https://schema.org")

def clean_node(node)->str:
    return str(node).split("/")[-1]
def remove_schema(part)->str:
    return str(part).replace("https://schema.org", "")

In [2]:
g = Graph()
g.parse("biosamples.jelly", format="jelly")

<Graph identifier=Ncf9edf6e0aa84d1989da22754bd622ab (<class 'rdflib.graph.Graph'>)>

In [74]:
res = g.query("SELECT ?s ?p ?o WHERE {?s ?p ?o. FILTER(!isBlank(?s))} LIMIT 50", initNs={"schema": SCHEMA})
for row in res:
    print(row)

(rdflib.term.URIRef('file:///home/japlan001/PycharmProjects/metadata_converter/output/uplifted/biosamples/Action_SAMEA112572036.jsonld'), rdflib.term.URIRef('https://schema.orgadditionalType'), rdflib.term.Literal('sampling process'))
(rdflib.term.URIRef('file:///home/japlan001/PycharmProjects/metadata_converter/output/uplifted/biosamples/Action_SAMEA118673944.jsonld'), rdflib.term.URIRef('https://schema.orgadditionalProperty'), rdflib.term.BNode('N6618531f05df4e4f864f0b3524201681'))
(rdflib.term.URIRef('file:///home/japlan001/PycharmProjects/metadata_converter/output/uplifted/biosamples/Product_SAMEA112558508.jsonld'), rdflib.term.URIRef('https://schema.orgurl'), rdflib.term.Literal('https://identifiers.org/biosample/SAMEA112558508'))
(rdflib.term.URIRef('file:///home/japlan001/PycharmProjects/metadata_converter/output/uplifted/biosamples/Action_SAMEA112587987.jsonld'), rdflib.term.URIRef('https://schema.orgadditionalProperty'), rdflib.term.BNode('N073d99a0413f4567bf137dbc5130a125'))


## General Statistics

In [50]:
print(f"{len(g):,}")

1,128,905


In [63]:
res = g.query(
    """
    SELECT (COUNT(DISTINCT ?s) AS ?cnt)
    WHERE {
    ?s ?p ?o .
    FILTER(!isBlank(?s))
    }
    """, initNs={"schema": SCHEMA, "rdf": RDF})


13475:,


In [67]:
print(f"{int(next(iter(res)).cnt):,}")

13,475


In [71]:
res = g.query(
    """
    SELECT ?type (COUNT(*) AS ?cnt)
    WHERE {
    ?s rdf:type ?type .
    }
    GROUP BY ?type
    """, initNs={"schema": SCHEMA, "rdf": RDF})

rows = [(remove_schema(row.type), int(row.cnt)) for row in res]
print(tabulate(rows, headers=["Type", "Count"]))

Type               Count
---------------  -------
ResearchProject    12705
PropertyValue     129719
Action              6721
DefinedTerm        27399
Product            16590
HowTo               4464
GeoCoordinates      6711
Place               6721
HowToStep           6977
MonetaryGrant          1


## Identify Missing values

### Empty Strings in Graph
Here I count how many strings are missing for which type

In [68]:
res = g.query(
    """
    SELECT ?p (COUNT(*) AS ?cnt)
    WHERE {?s ?p '' .}
    GROUP BY ?p ?cnt
    """,
    initNs={"schema": SCHEMA, "rdf": RDF}
)
rows = [(remove_schema(row.p), int(row.cnt)) for row in res]
print(tabulate(rows, headers=["Property", "Empty String Count"]))

Property       Empty String Count
-----------  --------------------
value                         608
description                   223


### Any Invalid Marker
Currently I look for empty strings and Literals containing the word "unknown" in both caps.

In [44]:
res = g.query(
    """
    SELECT ?type ?p ?o (COUNT(*) AS ?cnt)
    WHERE {
        ?s ?p ?o .
        ?s rdf:type ?type .
        FILTER(
            ?o = "" || REGEX(STR(?o), "unknown", "i")
        )
    }
    GROUP BY ?type ?p ?o ?cnt
    """,
    initNs={"schema": SCHEMA, "rdf": RDF}
)
rows = [(remove_schema(row.type), remove_schema(row.p), f'""' if row.o==rdflib.term.Literal('') else row.o, row.cnt) for row in res]
print(tabulate(rows, headers=["Node Type", "Property", "Object", "Count"]))


Node Type       Property     Object                                                                                                                                   Count
--------------  -----------  -------------------------------------------------------------------------------------------------------------------------------------  -------
PropertyValue   value        ""                                                                                                                                         608
Product         description  ""                                                                                                                                         223
GeoCoordinates  latitude     missing: control sample Unit unknown                                                                                                       228
GeoCoordinates  longitude    missing: control sample Unit unknown                                                                           

In [37]:
row.o

rdflib.term.Literal('')

In [ ]:
print(row.o)